In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F # using it to build neural networks without maintaining internal weights and biases

### Lets Define some Random Data ###
batch_size = 4 # processing in batches instead of one by one
sequence_length = 64 
emb_dim = 128 # Embedded dimension

x = torch.randn(batch_size, sequence_length, emb_dim)
print(f"shape of the input is: {x.shape}")

shape of the input is: torch.Size([4, 64, 128])



Implement Attention Without Any Learnable Parameters 
 
This whole attention operation is again very flexible and there is technically no reason to have any learnable parameters in its formulation (other than the obvious for wanting better predictive performance). So lets quickly just implement the formula as is using raw inputs 
 rather than doing any learned projections of the data.

Step 1:
Compute 
 which will provide the similarity score between every pair of vectors in 
. This will be contained inside a batch x sequence_length x sequence_length matrix

Step 2

After computing the similarity score, we can check and see that the variance of the similarity matrix is extremely high, this is the main reason for the normalization of dividing by the square root of the embedding dimension. In the end, the similarity scores are passed through a softmax to compute a probability vector, dividing by a constant basically acts as a temperature parameter to cool the distribution and provide more stable training!

Step 3

Each row of our sequence_length x sequence_length matrix is the similarity of how one timestep is related to all other timesteps! What we want to do is, instead of raw similarity scores, we will convert them to probabilities, so when we do the weighted average on our values matrix, the weights add up to 1!

In [ ]:
### First compute XX^T for similarity score betweeen every pair of tokens ###
similarity = (x @ x.transpose(1,2))

### Normalize the similarity scores ###
print("Prenormalized Variance:", similarity.var())
similarity_norm = similarity / (emb_dim ** 0.5)
print("Normed Similarity Variance:", similarity_norm.var())

### Check the shape of our similarity tensor ###
print("shape of Normed Similarity:", similarity_norm.shape)

### Compute similarity on every row of the attention matrix (i.e along the last dimension) ###
attention_mat = similarity_norm.softmax(dim=-1) # the -1 means the last dimension

### Verifying each row adds up to 1 ###
summed_attention_mat = similarity_norm.sum(axis=-1)
# To check if both tensors are equal and will return a boolean btw
print("Everything Equal to One:", torch.allclose(summed_attention_mat, torch.ones_like(summed_attention_mat))) 

### Multiply our Attention Matrix against its Values (X in our case)
convex_vectors = attention_mat @ x # the @ means multiplication
print("Output Shape:", convex_vectors.shape) # To have the same size as the input

Prenormalized Variance: tensor(382.8027)
Normed Similarity Variance: tensor(2.9906)
shape of Normed Similarity: torch.Size([4, 64, 64])
Everything Equal to One: False
Output Shape: torch.Size([4, 64, 128])



Thats it! This is basically all the attention computation is doing mathematically, we will add in the learnable parameters in just a bit. Something important to bring to your attention is the input shape of 
 and our output shape of the context vectors are identical. Again, the input is the raw data, the output is weighted averaged of how every token is realted to all the other ones. But the shapes not changing is quite convenient, and allows us to stack together a bunch attention mechanisms on top of one another

Lets Add Learnable Parameters
This time, instead of using 
 as our input, we will create our three projections of 
 (Queries, Keys and Values), but then repeat the operation we just did here! Also for convenience, I will wrap it all in a PyTorch class so we can continue adding stuff onto it as we go on!

Now what are these projections exactly? The are pointwise (or per timestep) projections! Remember, in our example here, each timestep is encoded by a vector of size 128. We will create three learnable weight matricies, incorporated inside the Linear modules in PyTorch, that take these 128 numbers per timestep and projects them to another 128 numbers (it is typical to keep the embedding dimension the same). This is a per timestep operation not across timesteps operation (across timesteps occurs within the attention computation). Obviously, PyTorch will accelerate this per timestep operation by doing it in parallel, but regardless, different timesteps dont get to see each other in the projection step.



In [25]:
class Attention(nn.Module):
    def __init__(self, embedding_dimension):
        super().__init__()

        self.embed_dim = embedding_dimension 
        
        ### Create Pointwise Projections ###
        self.query = nn.Linear(embedding_dimension, embedding_dimension)
        self.key = nn.Linear(embedding_dimension, embedding_dimension)
        self.value = nn.Linear(embedding_dimension, embedding_dimension)

    def forward(self, x):

        ### Create Queries, Keys and Values from X ###
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        ### Do the same Computation from above, just with our QKV Matricies instead of X ###
        similarity = (q @ k.transpose(1,2)) / (self.embed_dim ** 0.5)
        attention  = similarity.softmax(axis=-1)
        output = attention @ v

        return output

attention = Attention(embedding_dimension=128)
output = attention(x)
print(output.shape)

torch.Size([4, 64, 128])


MultiHeaded Attention
Now we have a small problem! Remember, the Attention matrix encodes the similarity between each pair of timesteps in your sequence. But in many cases, language being a prime example, there can be different types of relationships between different pairs of words, but our attention computation is restricted to only learn one of them. The solution to this is MultiHeaded Attention. Inside each attention computation, what if we have 2 attention matricies, or 8 or however many we want! The more we have the larger diversity of relationships we can learn!

Single Headed Attention Recap
Lets summarize everything we have seen so far with a single visual, and we will call this a Single Head of attention. We will also have our embedding dimension for each word in the sequence be 9, and the sequence length is 8.

It is general practice to have the number of heads you pick to be a divisor of the embedding dimension. For example, in our case, our original embedding dimension is 9, so we can pick 3 heads because 9 is divisible by 3. This also means our head dimension would now be 3, because 9/3 = 3. In typical transformers, the embedding dimension is 768, and they typically have 12 heads of attention. This means each head of attention will have a dimension of 64 because 768/12 = 64

The main reason we want it to evenly divide is because we have three heads, each takes in an embedding dimension of 9 and compresses to 3 before computing attention, and then outputs a tensor of embedding size 3. We can then take our 3 tensors, each having an embedding dimension of 3, concatenate them together, returning us back to the 9 that we began with! Again, this is just for convenience, so the embedding dimension of the input and output tensor dont change in any way. Last problem is, each head of attention is computed individually, so the final concatenated tensor has a bunch of heads of attention packed together, but we never got to share information between the different heads of attention. This is why we have the final head projection, that will take in the embedding dimension of 9 from our concatenated tensor, and output an embedding dimension of 9, therefore meshing information across the heads of our embedding dimension.

In [60]:

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        ### Make sure Embedding Dimension is Divisible by Num Heads ###
        assert embed_dim % num_heads == 0, f"Make sure your embed_dim {self.embed_dim} is divisible by the number of heads {self.num_heads}"

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        ### Compute Head Dimension ###
        self.head_dim = self.embed_dim // self.num_heads


        ### Create a List of Lists which has all our Q,K,V projections for each head ###
        self.multihead_qkv = nn.ModuleList()

        ### For head Head create the QKV ###
        for head in range(self.num_heads):

            ### Create a dictionary of the 3 projection  layers we need ###
            qkv_proj = nn.ModuleDict(
                [
                    ["Q", nn.Linear(self.embed_dim, self.head_dim)],
                    ["K", nn.Linear(self.embed_dim, self.head_dim)],
                    ["V", nn.Linear(self.embed_dim, self.head_dim)],
                ]
            )

            ### Store Dictionary in List ###
            self.multihead_qkv.append(qkv_proj)

        ### Create final Projection layer, it will be applied to the concatenated heads will have shape embed_dim again ###
        self.head_mesh = nn.Linear(self.embed_dim, self.embed_dim)
        
    def forward(self, x):

        ### Create a list ot store each heads output ###
        head_outs = []
        
        ### Loop Through Each head of Attention ###
        for head in self.multihead_qkv:

            ### Access layers like a dictionary (ModuleDict) ###
            ### q,k,v will be (Batch x Seq len x head_dim)
            q = head["Q"](x)
            k = head["K"](x)
            v = head["V"](x)

            ### Now do the same Attention computation as before! ###
            similarity = (q @ k.transpose(1,2)) / (self.embed_dim ** 0.5)
            attention  = similarity.softmax(axis=-1)
            output = attention @ v

            ### Store this output in the head_outs ###
            head_outs.append(output)

        ### head_outs has num_heads tensors, each with the compressed embedding dimension of head_dim ###
        ### We can concatenate them all back together along the embedding dimension just like we did in the image above ###
        head_outs = torch.cat(head_outs, dim=-1)

        ### head_outs will have the same shape now as our input x! ###
        if head_outs.shape != x.shape:
            raise Exception("Something has gone wrong in the attention computation")

        ### Now each head was computed independently, we need them to get to know each other, so pass our head_outs through final projection ### 
        output = self.head_mesh(head_outs)

        return output
        

embed_dim = 9
num_heads = 3
seq_len = 8
mha = MultiHeadAttention(embed_dim, num_heads)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
rand = torch.randn(3,seq_len,embed_dim)

### Pass through MHA ###
output = mha(rand)
print(output.shape)

torch.Size([3, 8, 9])


Increasing Efficiency:

If we want to increase efficiency then it is best to avoid using many loops or loops in general.

Linear Layers in MultiDimensional Tensors:

The Linear Layers applies only to the last dimension of the tensors.


In [63]:
function = nn.Linear(10, 30)

tensor1 = torch.randn(5, 10)
output_of_tensor1 = function(tensor1) # The first input of the linear function must be equal to the last input of the random number generated
print(f"Input Shape: {tensor1.shape}", f"Output Shape: {output_of_tensor1.shape}")

# Now we apply similar operation but with multiple dimension
tensor2 = torch.randn([5,1,2,3,4,10])
output_of_tensor2 = function(tensor2)
print(f"Input Shape: {tensor2.shape}", f"Output Shape: {output_of_tensor2.shape}")


Input Shape: torch.Size([5, 10]) Output Shape: torch.Size([5, 30])
Input Shape: torch.Size([5, 1, 2, 3, 4, 10]) Output Shape: torch.Size([5, 1, 2, 3, 4, 30])


PACKING LINEAR LAYERS:

In [67]:
tensor = torch.randn(1,8,9) # BATCH X SEQ_LENGTH X OUTPUT
fc = nn.Linear(9, 9) # INPUT X OUTPUT in other words the linear function
# Passing the tensor through the linear function to make queries
q = fc(tensor)
print(f"Shape of all Queries: {q.shape}")

# Cut the embedding dimensions into 3 heads 
q_head1, q_head2, q_head3 = torch.chunk(q, 3, axis=-1)
print(f"Shape of each Head of Query: {q_head1.shape}")


Shape of all Queries: torch.Size([1, 8, 9])
Shape of each Head of Query: torch.Size([1, 8, 3])


MultiDimensional Matrix Multiplication

We composed 9 linear layers into just 3 layers and we packed all the heads together

In [68]:
a = torch.randn(1,2,6,4)
b = torch.randn(1,2,4,3)
print(f" Final output Shape: {(a@b).shape}")

 Final output Shape: torch.Size([1, 2, 6, 3])


THE TRICK OF PARALLELIZING HEADS

Changing the [Batch x Seq_len x Embed_dim] --> [Batch x Seq_len x Num_Heads x Embed_dim]

In [82]:
class SelfAttentionEncoder(nn.Module):
    """
    Self Attention from the Attention is all you need paper
    """
    def __init__(self, embed_dim=768, num_heads=12, attn_p=0, proj_p=0):



        super(SelfAttentionEncoder, self).__init__()
        # Making sure that the number of embedding dimension is divisible by the num of heads
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = int (embed_dim / num_heads)



        # Define all projections
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_p)

        # Attention projection
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(proj_p)

    def forward(self, x):

        batch, seq_len, embed_dim = x.shape


        # Computing the projection of Q,K,V Reshape or applying permutation to them
        q = self.q_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)
        v = self.v_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim)


        # Performing the attention calculation
        attn = (q @ k.transpose(-2,-1)) * (self.head_dim ** 0.5)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = attn @ v

        # Return to it's previous shape
        x = x.transpose(1,2).reshape(batch, seq_len, embed_dim)

        # Pass through projection so head to know each othe
        x = self.proj(x)
        x = self.proj_drop(x)

        return x



embed_dim = 9
num_heads = 3
seq_len = 8

a = SelfAttentionEncoder(embed_dim, num_heads)
rand = torch.randn(3, seq_len, embed_dim)
output = a(rand)
print(f"Final Output: {output.shape}")

Final Output: torch.Size([3, 8, 9])


In [116]:
# where b is the batch and the n x n is the matrix
### Create an example attention matrix (b x n x n) ###
rand_attn = torch.rand(1,6,6)

# We then create an attention mask in the shape (b x n)
attention_mask = torch.tensor([1,1,1,1,0,0]).unsqueeze(0).bool()
print("Method 1:")
print("--------")
### Add Extra Dimension for the (b x n x n) ###
### So unsqueeze mask to be (b x 1 x n) ###
attention_mask = attention_mask.unsqueeze(1) # In order to match the size
print(attention_mask)
print(rand_attn.masked_fill_(~attention_mask, float("-inf")))


print("Method 2:")
print("--------")
### Repeat the Dummy Dimension so attention mask is (b x n x n) ###
attention_mask = attention_mask.repeat(1,6,1) # repeat dummy middle dim 6 times (for the seq_len) 
print(attention_mask)
print(rand_attn.masked_fill_(~attention_mask, float("-inf")))

Method 1:
--------
tensor([[[ True,  True,  True,  True, False, False]]])
tensor([[[0.5983, 0.6496, 0.8303, 0.0949,   -inf,   -inf],
         [0.4846, 0.9074, 0.6685, 0.3233,   -inf,   -inf],
         [0.5267, 0.5460, 0.5151, 0.9816,   -inf,   -inf],
         [0.5803, 0.6850, 0.6983, 0.9697,   -inf,   -inf],
         [0.1457, 0.6958, 0.8797, 0.2921,   -inf,   -inf],
         [0.5025, 0.6647, 0.6998, 0.3263,   -inf,   -inf]]])
Method 2:
--------
tensor([[[ True,  True,  True,  True, False, False],
         [ True,  True,  True,  True, False, False],
         [ True,  True,  True,  True, False, False],
         [ True,  True,  True,  True, False, False],
         [ True,  True,  True,  True, False, False],
         [ True,  True,  True,  True, False, False]]])
tensor([[[0.5983, 0.6496, 0.8303, 0.0949,   -inf,   -inf],
         [0.4846, 0.9074, 0.6685, 0.3233,   -inf,   -inf],
         [0.5267, 0.5460, 0.5151, 0.9816,   -inf,   -inf],
         [0.5803, 0.6850, 0.6983, 0.9697,   -inf,   -i

Matching Shapes for MultiHead Attentio

In [133]:
# Create an example attention matrix (b x h x n x n) ###
rand_attn = torch.rand(1,2,6,6) # I have 2 heads here!
### Create Attention Mask in the shape (b x n) ###
attention_mask = torch.tensor([1,1,1,1,0,0]).unsqueeze(0).bool()

print("Method 1:")
print("--------")
### Add Two Extra Dimension for the (b x h x n x n) ###
### So unsqueeze mask to be (b x 1 x 1 x n) ###
attention_mask = attention_mask.unsqueeze(1).unsqueeze(1)

# Since we mask it three times three columns will be masked 
### Unsqueezed with dummy broadcast dimension ###
print(attention_mask)
print(rand_attn.masked_fill_(~attention_mask, float("-inf")))

print("Method 2:")
print("--------")
### Repeat the Dummy Dimension for seq_len so attention mask is (b 1 x n x n) 

attention_mask = attention_mask.repeat(1,1,6,1)
print(attention_mask)
print(rand_attn.masked_fill_(~attention_mask, float("inf")))

Method 1:
--------
tensor([[[[ True,  True,  True,  True, False, False]]]])
tensor([[[[0.8495, 0.6501, 0.6814, 0.8493,   -inf,   -inf],
          [0.3652, 0.5442, 0.9678, 0.0251,   -inf,   -inf],
          [0.5271, 0.5138, 0.4771, 0.4005,   -inf,   -inf],
          [0.9097, 0.5446, 0.2958, 0.3119,   -inf,   -inf],
          [0.6113, 0.5500, 0.5467, 0.1556,   -inf,   -inf],
          [0.9065, 0.8452, 0.2832, 0.4580,   -inf,   -inf]],

         [[0.1762, 0.0387, 0.2049, 0.0987,   -inf,   -inf],
          [0.6684, 0.6430, 0.1554, 0.6989,   -inf,   -inf],
          [0.3406, 0.6386, 0.6324, 0.9978,   -inf,   -inf],
          [0.8515, 0.6598, 0.5737, 0.2905,   -inf,   -inf],
          [0.4094, 0.4682, 0.7357, 0.7829,   -inf,   -inf],
          [0.4297, 0.2428, 0.9012, 0.5285,   -inf,   -inf]]]])
Method 2:
--------
tensor([[[[ True,  True,  True,  True, False, False],
          [ True,  True,  True,  True, False, False],
          [ True,  True,  True,  True, False, False],
          [ True, 

In [136]:
class SelfAttention(nn.Module):
    """
    Self Attention Proposed in `Attention is All  You Need` - https://arxiv.org/abs/1706.03762
    """
    
    def __init__(self,
               embed_dim=768,
               num_heads=12, 
               attn_p=0,
               proj_p=0,
               causal=False):
        """
        
        Args:
            embed_dim: Transformer Embedding Dimension
            num_heads: Number of heads of computation for Attention 
            attn_p: Probability for Dropout2d on Attention cube
            proj_p: Probability for Dropout on final Projection
            causal: Do you want to apply a causal mask?
        """
        
        super(SelfAttention, self).__init__()

        ### Make Sure Embed Dim is Divisible by Num Heads ###
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = int(embed_dim / num_heads)
        self.causal = causal
        
        ### Define all our Projections ###
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_p)

        ### Define Post Attention Projection ###
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(proj_p)
        
    def forward(self, x, attention_mask=None):
        
        batch, seq_len, embed_dim = x.shape

        ### Compute Q, K, V Projections,and Reshape/Permute to [Batch x Num Heads x Seq Len x Head Dim] 
        q = self.q_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        k = self.k_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        v = self.v_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        
        ### Perform Attention Computation ###
        attn = (q @ k.transpose(-2,-1)) * (self.head_dim ** -0.5)

        if self.causal:
            ####################################################################################
            ### Create the Causal Mask (On the correct device) ###

            ### Create a Seq_Len x Seq_Len tensor full of Ones
            ones = torch.ones((seq_len, seq_len), device=attn.device)
            
            ### Fill Top right triangle with Zeros (as we dont want to attend to them) ###
            causal_mask = torch.tril(ones) 

            ### Add extra dimensions for Batch size and Number of Heads ###
            causal_mask = causal_mask.reshape(1,1,seq_len,seq_len).bool()

            ### If we have padding mask, then update our causal mask ###
            if attention_mask is not None:

                ### Each sample could have a different number of pad tokens, so repeat causal mask for batch size ###
                causal_mask = causal_mask.repeat(batch, 1, 1, 1)

                ### Expand and repeat the Padding Mask (b x s) -> (b x 1 x s x s)###
                attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1,1,seq_len,1)

                ### Fill causal mask where attention mask is False with False (to ensure all padding tokens are masked out) ###
                causal_mask = causal_mask.masked_fill(~attention_mask, False)

            ### Fill attn with -inf wherever causal mask is False ###
            attn = attn.masked_fill(~causal_mask, float('-inf'))

        ####################################################################################
    
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = attn @ v
        
        ### Bring Back to [Batch x Seq Len x Embed Dim] ###
        x = x.transpose(1,2).reshape(batch, seq_len, embed_dim)

        ### Pass through Projection so Heads get to know each other ###
        x = self.proj(x)
        x = self.proj_drop(x)
        
        return x

### We will now have sequences of different lengths, identify the number of tokens in each sequence ###
seq_lens = [3,5,4]
embed_dim = 9
num_heads = 3
a = SelfAttention(embed_dim, num_heads, causal=True)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
### This will be a tensor upto the max(seq_lens) ###
rand = torch.randn(len(seq_lens),max(seq_lens),embed_dim)

### Create Attention Mask from the seq_lens (shortest sequences padded to the longest ###
masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in seq_lens], batch_first=True, padding_value=0).bool()
print("Attention Mask:")
print(masks)

### Pass through MHA ###
output = a(rand, attention_mask=masks)
print("Final Output:", output.shape)

Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
Final Output: torch.Size([3, 5, 9])


In [146]:
class CrossAttention(nn.Module):

    # Defining the variables from the Attention is all you need paper
    def __init__(self,
                 embed_dim = 768,
                 num_heads = 12,
                 attn_p = 0,
                 proj_p = 0):


        super(CrossAttention, self).__init__()

        # Error handling using assertion
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = int(embed_dim / num_heads)

        # Define all the projections cause its cube 
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_p)

        ### Define Post Attention Projection ###
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(proj_p)
        
    def forward(self, src, tgt, attention_mask=None):
        
        batch, src_seq_len, embed_dim = src.shape
        batch, tgt_seq_len, embed_dim = tgt.shape

    def forward(self, src, tgt,  attention_mask=None):

        # Assigning the different values from the src and tgt
        batch, src_seq_len, embed_dim = src.shape
        batch, tgt_seq_len, embed_dim = tgt.shape

        # Compute Q (on our target french)
        q = self.q_proj(tgt).reshape(batch, tgt_seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()

        ### Compute K, V (on src English) Projections ###
        k = self.k_proj(src).reshape(batch, src_seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        v = self.v_proj(src).reshape(batch, src_seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        ### Perform Attention Computation ###
        attn = (q @ k.transpose(-2,-1)) * (self.head_dim ** -0.5)

        # FILLING THE ATTENTION MASK WITH INFINITY
        # attn.shape --> (batch x num_heads x french_seq_len x english_seq_len)
        # mask.shape --> (batch x english_seq_len)

        # Then we have to expand mask (batch x english_seq_len) --> (batch x 1 x 1 x english_seq_len x 1)

        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1,1, tgt_seq_len,1)
            attn = attn.masked_fill(~attention_mask, float('-inf'))


        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = attn @ v

        # Returning to the original size [Batch x Seq_Len x Embed Dim]
        x = x.transpose(1,2).reshape(batch, tgt_seq_len, embed_dim)
 
        # Passing through the projection so that they kow each other
        x = self.proj(x)
        x = self.proj_drop(x)

        return x





english_seq_lens = [3, 5, 4]
french_seq_lens = [7, 6, 2]

embed_dim = 18
num_heads = 3

a = CrossAttention(embed_dim, num_heads)

rand_english = torch.randn(len(english_seq_lens), max(english_seq_lens), embed_dim)
rand_french = torch.randn(len(french_seq_lens),max(french_seq_lens),embed_dim)


english_masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in english_seq_lens], batch_first=True, padding_value=0).bool()
french_masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in french_seq_lens], batch_first=True, padding_value=0).bool()

print("English Attention Mask:")
print(english_masks)
print("French Attention Mask:")
print(french_masks)

### Pass through MHA ###
output = a(src=rand_english, tgt=rand_french, attention_mask=english_masks)
print("Final Output:", output.shape)

English Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
French Attention Mask:
tensor([[ True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True, False, False, False, False, False]])
Final Output: torch.Size([3, 7, 18])


Lets Add Flash Attention to our Self Attention!

In [147]:

class SelfAttention(nn.Module):
    """
    Self Attention Proposed in `Attention is All  You Need` - https://arxiv.org/abs/1706.03762
    """

    def __init__(self,
                 embed_dim=768,
                 num_heads=12, 
                 attn_p=0,
                 proj_p=0,
                 causal=False,
                 fused_attn=True):

        """
        
        Args:
            embed_dim: Transformer Embedding Dimension
            num_heads: Number of heads of computation for Attention 
            attn_p: Probability for Dropout2d on Attention cube
            proj_p: Probability for Dropout on final Projection
            causal: Do you want to apply a causal mask?
        """
        
        super(SelfAttention, self).__init__()

        ### Make Sure Embed Dim is Divisible by Num Heads ###
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = int(embed_dim / num_heads)
        self.causal = causal
        self.fused = fused_attn

    def forward(self, x, attention_mask=None):
        
        batch, seq_len, embed_dim = x.shape

        ### Compute Q, K, V Projections,and Reshape/Permute to [Batch x Num Heads x Seq Len x Head Dim] 
        q = self.q_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        k = self.k_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
        v = self.v_proj(x).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()


        if self.fused:

            x = F.scaled_dot_product_attention(q,k,v,
                                               is_causal=self.causal,
                                               attn_mask=attention_mask)

        else:

            ### Use Flash Attention ###
            attn = (q @ k.transpose(-2,-1)) * (self.head_dim ** 0.5)

            if self.causal:

                # Creat the Causal Mask (on the right device)

                # Creating a Seq_len x Seq_len tensor full of Ones
                ones = torch.ones((seq_len, seq_len), device=attn.device)

                # Fill top triangle with ones
                causal_mask = torch.tril(ones)

                # Add extra dimension for batch size and number of heads
                causal_mask = causal_mask.reshape(1,1,seq_len,seq_len).bool()


                # If we have padding mask then we update our causal mask
                if attention_mask is not None:

                    # Each sample can have different number of paddings so we add the padding for the
                    causal_mask = causal_mask.repeat(batch,1,1,1)

                    # We then expand and repeat the padding mask (BXS)--> (BX1XSXS)
                    attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1,1,seq_len,1)

                    # Fill the causal mask where the attention mask is false for false (ensuring padding tokens are masked out)
                    causal_mask = causal_mask.masked_fill(~attention_mask, False)

                # Now we do the same for the causal mask but with infinity
                attn = attn.masked_fill(~causal_mask, float('-inf'))


                attn = attn.softmax(dim=-1)
                attn = self.attn_proj(attn)

            # Returning it to the original size    
            x = x.transpose(1,2).reshape(batch, seq_len, embed_dim)

            # Finally passing it through the projections
            x = self.proj(x)
            x = self.proj_drop(x)

            return x





  


        

Attention Mask for Cross Attention

In [148]:
class Attention(nn.Module):
    """
    Regular Self-Attention but in this case we utilize flash_attention
    incorporated in the F.scaled_dot_product_attention to speed up our training. 
    """
    def __init__(self, embedding_dimension=768, num_heads=12, attn_dropout=0.0):
        super(Attention, self).__init__()

        self.embedding_dimension = embedding_dimension
        self.num_heads = num_heads
        self.attn_dropout = attn_dropout
        
        ### Sanity Checks ###
        assert embedding_dimension % num_heads == 0, "Double check embedding dim divisible by number of heads"

        ### Attention Head Dim ###
        self.head_dim = embedding_dimension // num_heads

        ### Attention Projections ###
        self.q_proj = nn.Linear(embedding_dimension, embedding_dimension)
        self.k_proj = nn.Linear(embedding_dimension, embedding_dimension)
        self.v_proj = nn.Linear(embedding_dimension, embedding_dimension)

        ### Post Attention Projection ###
        self.out_proj = nn.Linear(embedding_dimension, embedding_dimension)
        

    def forward(self, 
                src, 
                tgt=None, 
                attention_mask=None, 
                causal=False):

        """
        By default, self-attention will be computed on src (with optional causal and/or attention mask). If tgt is provided, then
        we are doing cross attention. In cross attention, an attention_mask can be used, but no causal mask can be applied.
        """

        ### Grab Shapes ###
        batch, src_len, embed_dim = src.shape

        ### If target is not provided, we are doing self attention (with potential causal mask) ###    
        if tgt is None:
            q = self.q_proj(src).reshape(batch, src_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
            k = self.k_proj(src).reshape(batch, src_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
            v = self.v_proj(src).reshape(batch, src_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()

            if attention_mask is not None:

                attention_mask = attention_mask.bool()
                attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1,1,src_len,1)

            attention_out = F.scaled_dot_product_attention(q,k,v, 
                                                           attn_mask=attention_mask, 
                                                           dropout_p=self.attn_dropout if self.training else 0.0, 
                                                           is_causal=causal)
        
        ### If target is provided then we are doing cross attention ###
        ### Our query will be the target and we will be crossing it with the encoder source (keys and values) ###
        ### The src_attention_mask will still be the mask here, just repeated to the target size ###
        else:
            tgt_len = tgt.shape[1]

            q = self.q_proj(tgt).reshape(batch, tgt_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
            k = self.k_proj(src).reshape(batch, src_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()
            v = self.v_proj(src).reshape(batch, src_len, self.num_heads, self.head_dim).transpose(1,2).contiguous()

            if attention_mask is not None:

                attention_mask = attention_mask.bool()
                attention_mask = attention_mask.unsqueeze(1).unsqueeze(1).repeat(1,1,tgt_len,1)

            attention_out = F.scaled_dot_product_attention(q,k,v,
                                                           attn_mask=attention_mask, 
                                                           dropout_p=self.attn_dropout if self.training else 0.0, 
                                                           is_causal=False)

        ### Reshape and Project ###
        attention_out = attention_out.transpose(1,2).flatten(2)
        attention_out = self.out_proj(attention_out)

        return attention_out

### Test Out Self-Attention!! ###
print("TESTING SELF-ATTENTION!!!")
print("-------------------------")
seq_lens = [3,5,4]
embed_dim = 9
num_heads = 3
a = Attention(embed_dim, num_heads)

### Create a random tensor in the shape (Batch x Seq Len x Embed Dim) ###
### This will be a tensor upto the max(seq_lens) ###
rand = torch.randn(len(seq_lens),max(seq_lens),embed_dim)

### Create Attention Mask from the seq_lens (shortest sequences padded to the longest ###
masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in seq_lens], batch_first=True, padding_value=0).bool()
print("Attention Mask:")
print(masks)

### Pass through MHA ###
output = a(rand, attention_mask=masks, causal=True)
print("Final Output:", output.shape, "\n")


print("TESTING CROSS-ATTENTION!!!")
print("-------------------------")
### Test out Cross Attention 
### We will now have sequences of different lengths, identify the number of tokens in each sequence ###
english_seq_lens = [3,5,4]
french_seq_lens = [7,6,2]

embed_dim = 9
num_heads = 3
a = Attention(embed_dim, num_heads)

### Create random tensor in the shape (Batch x Seq Len x Embed Dim) for French and English ###
### This will be a tensor upto the max(seq_lens) ###
rand_english = torch.randn(len(english_seq_lens),max(english_seq_lens),embed_dim)
rand_french = torch.randn(len(french_seq_lens),max(french_seq_lens),embed_dim)


### Create Attention Mask from the seq_lens (shortest sequences padded to the longest ###
english_masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in english_seq_lens], batch_first=True, padding_value=0).bool()
french_masks = torch.nn.utils.rnn.pad_sequence([torch.ones(l) for l in french_seq_lens], batch_first=True, padding_value=0).bool()

print("English Attention Mask:")
print(english_masks)
print("French Attention Mask:")
print(french_masks)

### Pass through MHA ###
output = a(src=rand_english, tgt=rand_french, attention_mask=english_masks)
print("Final Output:", output.shape)

TESTING SELF-ATTENTION!!!
-------------------------
Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
Final Output: torch.Size([3, 5, 9]) 

TESTING CROSS-ATTENTION!!!
-------------------------
English Attention Mask:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False]])
French Attention Mask:
tensor([[ True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True, False, False, False, False, False]])
Final Output: torch.Size([3, 7, 9])
